# Compare Duplicates

Perceptual-hash duplicate detection (dHash + Hamming distance). Three independent tools are provided below.

## Compare two folders

Flags images in folder B that match any image in folder A.

In [ ]:
import os
import cv2
import numpy as np
from pathlib import Path

def calculate_dhash(image_path, hash_size=8):
    img = cv2.imread(str(image_path))
    if img is None: return None
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, (hash_size + 1, hash_size))
    diff = resized[:, 1:] > resized[:, :-1]
    return sum([2**i for i, v in enumerate(diff.flatten()) if v])

def visual_compare_folders(path_a, path_b, output_preview_path, similarity_threshold=0.95):
    os.makedirs(output_preview_path, exist_ok=True)
    max_distance = int(64 * (1 - similarity_threshold))
    
    def get_images(path):
        return list(Path(path).rglob("*.[jJ][pP][gG]")) + \
               list(Path(path).rglob("*.[pP][nN][gG]"))

    images_a = get_images(path_a)
    images_b = get_images(path_b)
    
    print(f"Index: Hashing {len(images_a)} images from Path A...")
    hashes_a = {p: calculate_dhash(p) for p in images_a if calculate_dhash(p) is not None}

    print(f"Compare: Checking {len(images_b)} images from Path B...")
    match_count = 0

    for pb in images_b:
        hb = calculate_dhash(pb)
        if hb is None: continue
        
        for pa, ha in hashes_a.items():
            distance = bin(hb ^ ha).count('1')
            
            if distance <= max_distance:
                match_count += 1
                
                img_a = cv2.imread(str(pa))
                img_b = cv2.imread(str(pb))
                
                h_new = 600
                img_a_res = cv2.resize(img_a, (int(img_a.shape[1] * (h_new / img_a.shape[0])), h_new))
                img_b_res = cv2.resize(img_b, (int(img_b.shape[1] * (h_new / img_b.shape[0])), h_new))
                
                canvas = np.zeros((h_new + 80, img_a_res.shape[1] + img_b_res.shape[1] + 20, 3), dtype=np.uint8)
                canvas[40:h_new+40, :img_a_res.shape[1]] = img_a_res
                canvas[40:h_new+40, img_a_res.shape[1]+20:] = img_b_res
                
                cv2.putText(canvas, f"PATH A: {pa.name}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
                cv2.putText(canvas, f"PATH B: {pb.name} (Dist: {distance})", (img_a_res.shape[1]+30, 30), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

                save_name = f"match_{match_count}_{pb.name}"
                cv2.imwrite(os.path.join(output_preview_path, save_name), canvas)
                break 

    print("-" * 40)
    print(f"Found {match_count} duplicates.")
    print(f"Verification images saved to: {output_preview_path}")

folder_1 = r"D:\Desktop\Sesame Datasets\Roboflow (test-mrio3)\Powdery Mildew"
folder_2 = r"D:\Desktop\Sesame Datasets\Roboflow (new-workspace-mwunv)\Powdery Mildew"
preview_dir = r"D:\Desktop\Deduplication_Visual_Check"

visual_compare_folders(folder_1, folder_2, preview_dir)

## Find duplicates within one folder

Scans a single folder for internal duplicates.

In [ ]:
import os
import cv2
import numpy as np
from pathlib import Path

def calculate_dhash(image_path, hash_size=8):
    img = cv2.imread(str(image_path))
    if img is None: return None
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, (hash_size + 1, hash_size))
    diff = resized[:, 1:] > resized[:, :-1]
    return sum([2**i for i, v in enumerate(diff.flatten()) if v])

def find_internal_duplicates(root_folder, output_preview_path, similarity_threshold=0.95):
    """Scans one folder for duplicates and saves visual comparisons."""
    os.makedirs(output_preview_path, exist_ok=True)
    max_distance = int(64 * (1 - similarity_threshold))
    
    image_paths = list(Path(root_folder).rglob("*.[jJ][pP][gG]")) + \
                  list(Path(root_folder).rglob("*.[pP][nN][gG]"))
    
    hashes = {}
    match_count = 0

    print(f"Scanning {len(image_paths)} images for internal duplicates...")

    for i, path in enumerate(image_paths):
        h = calculate_dhash(path)
        if h is None: continue
        
        is_duplicate = False
        for existing_path, existing_hash in hashes.items():
            distance = bin(h ^ existing_hash).count('1')
            
            if distance <= max_distance:
                match_count += 1
                is_duplicate = True
                
                img_orig = cv2.imread(str(existing_path))
                img_dupe = cv2.imread(str(path))
                
                h_new = 600
                img_a = cv2.resize(img_orig, (int(img_orig.shape[1] * (h_new / img_orig.shape[0])), h_new))
                img_b = cv2.resize(img_dupe, (int(img_dupe.shape[1] * (h_new / img_dupe.shape[0])), h_new))
                
                canvas = np.zeros((h_new + 80, img_a.shape[1] + img_b.shape[1] + 20, 3), dtype=np.uint8)
                canvas[40:h_new+40, :img_a.shape[1]] = img_a
                canvas[40:h_new+40, img_a.shape[1]+20:] = img_b
                
                cv2.putText(canvas, f"ORIGINAL: {existing_path.name}", (10, 30), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
                cv2.putText(canvas, f"DUPLICATE: {path.name} (Dist: {distance})", (img_a.shape[1]+30, 30), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

                save_name = f"dupe_{match_count}_{path.name}"
                cv2.imwrite(os.path.join(output_preview_path, save_name), canvas)
                break
        
        if not is_duplicate:
            hashes[path] = h
        
        if i % 100 == 0:
            print(f"Processed {i}/{len(image_paths)} images...")

    print("-" * 40)
    print(f"Found {match_count} duplicates.")
    print(f"Verification images saved to: {output_preview_path}")

master_folder = r"D:\Desktop\Sesame Datasets\Roboflow (z-combined)\Powdery Mildew"
preview_folder = r"D:\Desktop\Duplicate_Visual_Check"

find_internal_duplicates(master_folder, preview_folder)

## Build a duplicate-free copy

Copies only unique images into a new folder.

In [ ]:
import os
import cv2
import numpy as np
import shutil
from pathlib import Path

def calculate_dhash(image_path, hash_size=8):
    img = cv2.imread(str(image_path))
    if img is None: return None
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, (hash_size + 1, hash_size))
    diff = resized[:, 1:] > resized[:, :-1]
    return sum([2**i for i, v in enumerate(diff.flatten()) if v])

def create_unique_dataset(source_folder, output_folder, similarity_threshold=0.95):
    os.makedirs(output_folder, exist_ok=True)
    
    max_distance = int(64 * (1 - similarity_threshold))
    
    image_paths = list(Path(source_folder).rglob("*.[jJ][pP][gG]")) + \
                  list(Path(source_folder).rglob("*.[pP][nN][gG]"))
    
    known_hashes = {}
    unique_count = 0
    duplicate_count = 0

    print(f"Scanning {len(image_paths)} images to build a unique set...")

    for i, path in enumerate(image_paths):
        h = calculate_dhash(path)
        if h is None: continue
        
        is_duplicate = False
        for existing_hash in known_hashes.values():
            distance = bin(h ^ existing_hash).count('1')
            
            if distance <= max_distance:
                is_duplicate = True
                duplicate_count += 1
                break
        
        if not is_duplicate:
            known_hashes[path] = h
            unique_count += 1
            
            shutil.copy2(path, os.path.join(output_folder, path.name))
        
        if i % 100 == 0:
            print(f"Processed {i}/{len(image_paths)}... Found {unique_count} unique so far.")

    print("-" * 40)
    print(f"PURIFICATION COMPLETE")
    print(f"Original Images: {len(image_paths)}")
    print(f"Unique Images Kept: {unique_count}")
    print(f"Duplicates Skipped: {duplicate_count}")
    print(f"Your clean dataset is here: {output_folder}")

source_dir = r"D:\Desktop\Sesame_FINAL_UNIQUE"
clean_dir = r"D:\Desktop\Sesame_FINAL_UNIQUE2"

create_unique_dataset(source_dir, clean_dir)